# <center> EDA</center>


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
from scipy import stats
warnings.filterwarnings("ignore")
%matplotlib inline

# Import the dataset and display the first 10 rows. Also print the shape of the dataset.

In [4]:
df=pd.read_csv("Auto_Sales.csv")

In [5]:
df.head(10)

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,SALES,ORDERDATE,DAYS_SINCE_LASTORDER,STATUS,PRODUCTLINE,MSRP,PRODUCTCODE,CUSTOMERNAME,ADDRESSLINE1,CITY,POSTALCODE,COUNTRY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2871.00,24-02-2018,828,Shipped,Motorcycles,95,S10_1678,Land of Toys Inc.,897 Long Airport Avenue,NYC,10022,USA,Yu,Kwai,Small
1,10121,34,81.35,2765.90,07-05-2018,757,Shipped,Motorcycles,95,S10_1678,Reims Collectables,59 rue de l'Abbaye,Reims,51100,France,Henriot,Paul,Small
2,10134,41,94.74,3884.34,01-07-2018,703,Shipped,Motorcycles,95,S10_1678,Lyon Souveniers,27 rue du Colonel Pierre Avia,Paris,75508,France,Da Cunha,Daniel,Medium
3,10145,45,83.26,3746.70,25-08-2018,649,Shipped,Motorcycles,95,S10_1678,Toys4GrownUps.com,78934 Hillside Dr.,Pasadena,90003,USA,Young,Julie,Medium
4,10168,36,96.66,3479.76,28-10-2018,586,Shipped,Motorcycles,95,S10_1678,Technics Stores Inc.,9408 Furth Circle,Burlingame,94217,USA,Hirano,Juri,Medium
5,10180,29,86.13,2497.77,11-11-2018,573,Shipped,Motorcycles,95,S10_1678,Daedalus Designs Imports,"184, chausse de Tournai",Lille,59000,France,Rance,Martine,Small
6,10188,48,114.84,5512.32,18-11-2018,567,Shipped,Motorcycles,95,S10_1678,Herkku Gifts,"Drammen 121, PR 744 Sentrum",Bergen,N 5804,Norway,Oeztan,Veysel,Medium
7,10211,41,114.84,4708.44,15-01-2019,510,Shipped,Motorcycles,95,S10_1678,Auto Canal Petit,"25, rue Lauriston",Paris,75016,France,Perrier,Dominique,Medium
8,10223,37,107.18,3965.66,20-02-2019,475,Shipped,Motorcycles,95,S10_1678,"Australian Collectors, Co.",636 St Kilda Road,Melbourne,3004,Australia,Ferguson,Peter,Medium
9,10237,23,101.44,2333.12,05-04-2019,432,Shipped,Motorcycles,95,S10_1678,Vitachrome Inc.,2678 Kingston Rd.,NYC,10022,USA,Frick,Michael,Small


In [ ]:
df.shape

In [ ]:
#Checking the data types of all columns.

In [ ]:
df.info()

In [ ]:
#Checking for any duplicates records and delete them.

In [ ]:
df.duplicated().sum()

In [ ]:
#Checking for the outliers in all numerical variables with IQR method.

In [ ]:
id_cols = ["ORDERNUMBER","POSTALCODE"]

num_cols = df.select_dtypes(include="number").columns.tolist()
num_cols = [col for col in num_cols if col not in id_cols]
num_cols

In [ ]:
outlier_summary = {}

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    Lower = Q1 - 1.5*IQR
    Upper = Q3 + 1.5*IQR
    outliers = df[(df[col] < Lower) | (df[col] > Upper)]

    outlier_summary[col] = {
        "Lower_Bound":Lower,
        "Upper_Bound":Upper,
        "Outlier_counts":outliers.shape[0],
        "Outlier_percentage":round((outliers.shape[0]/df.shape[0])*100,2)
    }

pd.DataFrame(outlier_summary).T

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6, 4))
    plt.boxplot(df[col], vert=False)
    plt.title(f'Boxplot of {col}')
    plt.xlabel(col)
    plt.show()

In [ ]:
#Find the total number of unique customers and unique countries.

In [ ]:
unique_customers = df["ORDERNUMBER"].nunique()
unique_countries = df["COUNTRY"].nunique()

print(f"Total unique customers: {unique_customers}")
print(f"Total unique countries: {unique_countries}")

In [ ]:
#Convert ORDERDATE into a datetime format. Extract Year and Month as new columns.

In [ ]:
df["ORDERDATE"]=pd.to_datetime(df["ORDERDATE"],errors="coerce",dayfirst=True)
df["MONTH"]=df["ORDERDATE"].dt.month_name()
df["YEAR"]=df["ORDERDATE"].dt.year

In [ ]:
#Create a visualization showing monthly sales trend over time.

In [ ]:
monthly_sales = df.groupby(pd.Grouper(key='ORDERDATE', freq='M'))['SALES'].sum()

plt.figure(figsize=(10, 4))
plt.plot(monthly_sales.index, monthly_sales.values)
plt.xlabel('Month')
plt.ylabel('Total Sales')
plt.title('Monthly Sales Trend Over Time')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#Find the total sales by country and identify the top 3 countries by sales.

In [ ]:
Countries_Sales = df.groupby("COUNTRY")["SALES"].sum().sort_values(ascending=False).reset_index()
Top_3_Countries = Countries_Sales.head(3)
Top_3_Countries

In [ ]:
#Calculate the average order value (SALES) for each DEALSIZE. Which deal size contributes the most revenue.

In [ ]:
# Average order value (SALES) for each DEALSIZE
avg_sales_per_deal = df.groupby('DEALSIZE')['SALES'].mean()
print("Average SALES per DEALSIZE:\n", avg_sales_per_deal)

# Total revenue contribution by each DEALSIZE
total_revenue_per_deal = df.groupby('DEALSIZE')['SALES'].sum()
most_revenue_deal = total_revenue_per_deal.idxmax()

print("\nDEALSIZE contributing the most revenue:", most_revenue_deal)

In [ ]:
#Identify and count items where the selling price is higher than MSRP

In [ ]:
df.head(1)

In [ ]:
items_Selling_Price_higher_than_MSRP = df[df["PRICEEACH"]>df["MSRP"]].reset_index(drop=True)

count_items_Selling_Price_higher_than_MSRP = items_Selling_Price_higher_than_MSRP.shape[0]


print("\nCount of such items:", count_items_Selling_Price_higher_than_MSRP)
print("Items where selling price is higher than MSRP:")
items_Selling_Price_higher_than_MSRP

In [ ]:
#Create a bar chart showing total sales by product line.

In [ ]:
total_sales = df.groupby('PRODUCTLINE')['SALES'].sum()

plt.figure(figsize=(10,6))
bars = plt.bar(total_sales.index, total_sales.values, color='skyblue')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.01*yval, round(yval, 2), ha='center', va='bottom')

plt.title('Total Sales by Product Line')
plt.xlabel('Product Line')
plt.ylabel('Total Sales')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
#Find the relationship between QUANTITYORDERED, PRICEEACH, SALES. Create a correlation matrix and visualize it using a heatmap.

In [ ]:
corr_matrix = df[["QUANTITYORDERED","PRICEEACH","SALES"]].corr()
corr_matrix

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(corr_matrix,annot=True,cmap="magma",fmt=".2f")
plt.title("Heatmap")
plt.show()

In [ ]:
#Find customers whose last order was more than the average days using DAYS_SINCE_LASTORDER and print customer name and days since last order.

In [ ]:
df.loc[df["DAYS_SINCE_LASTORDER"]>df["DAYS_SINCE_LASTORDER"].mean(),["CUSTOMERNAME","DAYS_SINCE_LASTORDER"]].reset_index(drop=True)

In [ ]:
#Identify customers who have placed more than one order. Print customer name and number of orders

In [ ]:
orders = df.groupby("CUSTOMERNAME")["ORDERNUMBER"].nunique().reset_index(name="No of orders")
multiple_order_customer = orders[orders["No of orders"]>1]
multiple_order_customer

In [ ]:
#Check the missing records in each column.
df.isnull().sum()

In [ ]:
#Create 2 Dataframes – 1 for numerical and 1 for categorical using pandas
df_num = df.select_dtypes(include = 'number')
df_cat = df.select_dtypes(include = 'object')
df.info()

In [ ]:
df_num.dtypes

In [ ]:
df_cat.dtypes

In [ ]:
#Rename the ORDERNUMBER as order_number and QUANTITYORDERED as qty_ordered.
df = df.rename(columns = {'ORDERNUMBER' :'order_number','QUANTITYORDERED' : 'qty_ordered'})
df.columns

In [ ]:
#For the numerical dataframe created in Q16, perform the feature scaling (using standard scaler) to bring all variables on same scale.

In [ ]:
df_num_cols = df.select_dtypes(include='number').drop(columns=['order_number'])
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_num_cols)
scaled_data = pd.DataFrame(scaled_data, columns=df_num_cols.columns, index = df_num_cols.index)
scaled_data

In [ ]:
#Check for the skewness in all numerical variables and tag them as low skewed, moderately skewed and highly skewed.

In [ ]:
df_num_cols = df.select_dtypes(include='number').drop(columns=['order_number'])

skewness_data = df_num_cols.skew()

def skew_tag(skewness_value):
    x=abs(skewness_value)
    if x<0.5:
        return "Low_Skewed"
    elif x<1:
        return "Moderately_Skewed"
    else:
        return "Highly_Skewed"

skewness_data_summary = pd.DataFrame({"Skewness":skewness_data, "Skewness_type":skewness_data.apply(skew_tag)}).reset_index(names = "Numerical_Variables")
skewness_data_summary

In [ ]:
#Visualization of skewness

num_cols = df_num_cols.columns

for col in num_cols:
    plt.figure(figsize=(6, 4))
    plt.hist(df_num_cols[col], bins=30, density=True, edgecolor="black", color = "skyblue")
    df_num_cols[col].plot(kind='kde',color="green")
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Density")
    plt.show()


In [ ]:
#For highly skewed variables, perform the transformation techniques and make the less/moderately skewed.

In [ ]:
highly_skewed_cols = skewness_data_summary[skewness_data_summary["Skewness_type"]=="Highly_Skewed"]["Numerical_Variables"].tolist()
highly_skewed_cols

In [ ]:
df_num_transformed = df_num_cols.copy()

for col in highly_skewed_cols:
    df_num_transformed[col+'_log'] = np.log1p(df_num_transformed[col])
    df_num_transformed[col+'_sqrt'] = np.sqrt(df_num_transformed[col])
    
    boxcox_data = df_num_transformed[col]
    if (boxcox_data <= 0).any():
        boxcox_data = boxcox_data + 1
        
    df_num_transformed[col+'_boxcox'], lambda_opt = stats.boxcox(boxcox_data)
    
    print(f"Original {col} skewness: {round(df_num_transformed[col].skew(), 2)}")
    print(f"Transformed {col} skewness boxcox: {round(df_num_transformed[col+'_boxcox'].skew(), 2)}")
    print(f"Transformed {col} skewness log: {round(df_num_transformed[col+'_log'].skew(), 2)}")
    print(f"Transformed {col} skewness sqrt: {round(df_num_transformed[col+'_sqrt'].skew(), 2)}")
#Log, square-root, and Box–Cox transformations were applied to highly skewed variables, and skewness values were compared to assess which transformation best reduced skewness.